# 03. Cleaning and the Analytics Warehouse (dbt + DuckDB)

**Procurement Intelligence Platform** | Notebook 03 of 13

The cleaning rules from the audit become tested SQL models. The result is a star schema, a governed KPI layer and a feature store that every notebook, the dashboard and the AI agent share.

| | |
|---|---|
| **Inputs** | `data/raw/*.csv`, `warehouse/` (dbt project) |
| **Outputs** | `data/warehouse/p2p.duckdb`: 33 models, 2 seeds, 42 data tests, all passing |
| **Skills shown** | Analytics engineering: dbt, dimensional modelling, testing, documentation |

## 1. Architecture

| Layer | Purpose | Materialisation | Examples |
|---|---|---|---|
| **Sources (bronze)** | Raw CSV extracts read in place, all text | external | `sap_raw.purchase_orders` |
| **Staging (silver)** | One model per source: types, names, de-duplication, flags | view | `stg_purchase_requisitions`, `stg_goods_receipts` |
| **Intermediate** | Business logic: receipts, approvals, payments, the line lifecycle | table | `int_requisition_lifecycle`, `int_status_events` |
| **Marts (gold)** | Star schema for analysis and the app | table | `fct_requisition_line`, `dim_supplier`, `fct_backlog_weekly` |
| **KPIs** | Governed monthly KPI table with targets and status | table | `kpi_monthly_company` |
| **Feature store** | Leakage-safe ML features built with window functions | table | `ml_features_weekly` |

DuckDB is the engine: an embedded analytical database, so the whole warehouse is one file, runs in CI and needs no server.

In [1]:
import sys, warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from p2p.viz.theme import use_matplotlib, BRAND, SERIES, ORIGIN_COLORS, STATUS, shade_shocks
use_matplotlib()
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

In [2]:
import subprocess, os, re
env = {**os.environ, "P2P_RAW_DIR": str(ROOT / "data" / "raw"), "P2P_DUCKDB_PATH": str(ROOT / "data" / "warehouse" / "p2p.duckdb")}
run = subprocess.run(["dbt", "build", "--profiles-dir", "."], cwd=ROOT / "warehouse", env=env, capture_output=True, text=True)
log = re.sub(r"\x1b\[[0-9;]*m", "", run.stdout)
print("\n".join(l for l in log.splitlines() if "Finished running" in l or "Done." in l or "ERROR" in l or "FAIL " in l))

16:25:32  Finished running 2 seeds, 19 table models, 42 data tests, 14 view models in 0 hours 0 minutes and 4.53 seconds (4.53s).
16:25:32  Done. PASS=77 WARN=0 ERROR=0 SKIP=0 NO-OP=0 REUSED=0 TOTAL=77


**What we see.** `dbt build` runs every model and every test in dependency order: 2 seeds, 33 models and 42 data tests (77 nodes in total), all passing,
in a few seconds.

## 2. Staging: the audit rules as code

The staging model for requisitions removes exact duplicates and maps free-text section names through the governed seed:

In [3]:
print((ROOT / "warehouse" / "models" / "staging" / "stg_purchase_requisitions.sql").read_text())

-- One clean row per PR line: exact duplicates removed, section text mapped to a section id.
with source as (
    select * from {{ source('sap_raw', 'purchase_requisitions') }}
),
deduplicated as (
    select *
    from source
    qualify row_number() over (partition by pr_number, pr_item order by created_on) = 1
)
select
    cast(d.pr_number as bigint)                         as pr_number,
    cast(d.pr_item as integer)                          as pr_item,
    cast(d.pr_number as varchar) || '-' || d.pr_item    as pr_line_key,
    cast(d.created_on as date)                          as pr_created_on,
    cast(nullif(d.released_on, '') as date)             as pr_released_on,
    a.section_id,
    d.section                                           as section_raw_text,
    d.material_id,
    d.short_text,
    cast(d.quantity as integer)                         as quantity,
    d.uom,
    cast(d.estimated_price_usd as double)               as estimated_price_usd,
    cast(d.quantity as in

In [4]:
from p2p.data import query
before_after = query("""
    select section_raw_text, section_id, count(*) as lines
    from staging.stg_purchase_requisitions
    where section_id = 'SEC04'
    group by all order by lines desc
""")
display(before_after)

,section_raw_text,section_id,lines
0,Light Vehicles,SEC04,4085
1,Light Vehicle,SEC04,266
2,LIGHT VEHICLE,SEC04,221
3,light vehicles,SEC04,219


**What we see.** Six different spellings of "Light Vehicles" now resolve to the same section id `SEC04`. The raw text is kept next to
the governed id for traceability.

## 3. The lifecycle model: one row per requisition line

`int_requisition_lifecycle` joins requisitions, RFQs, POs, approvals, receipts and payments, applies the remaining cleaning rules and
derives the status **from the documents**, not from the SAP status field (which the audit showed to be unreliable).

In [5]:
status = query("""
    select lifecycle_status, sap_processing_status, count(*) as lines
    from marts.fct_requisition_line
    where is_open or sap_processing_status in ('Not Edited')
    group by all order by lines desc
""")
display(status.pivot_table(index="lifecycle_status", columns="sap_processing_status", values="lines", fill_value=0))

flags = query("""
    select sum(dq_missing_supplier::int) as missing_supplier, sum(dq_zero_price::int) as zero_price_repaired,
           sum(dq_po_date_corrected::int) as po_date_corrected, sum(dq_over_delivery::int) as over_delivery,
           sum(dq_future_posting_date::int) as future_date_repaired, sum(dq_status_mismatch::int) as sap_status_mismatch
    from marts.fct_requisition_line
""")
display(flags)

sap_processing_status,Closed,Not Edited,PO Created,RFQ Created
lifecycle_status,,,,
Awaiting Approval,0.000,0.000,1.000,0.000
Awaiting Delivery,0.000,11.000,"1,322.000",0.000
Awaiting Downpayment,0.000,0.000,31.000,0.000
Not Edited,0.000,303.000,0.000,0.000
RFQ Created,44.000,0.000,3.000,294.000


,missing_supplier,zero_price_repaired,po_date_corrected,over_delivery,future_date_repaired,sap_status_mismatch
0,171.000,72.000,69.000,117.000,6.000,11.000


**What we see.** The derived status splits the backlog more finely than SAP does ("Awaiting Delivery" versus "Awaiting Downpayment"),
and 11 lines that SAP marks "Not Edited" although a PO exists are now classified correctly as awaiting delivery.

The table also exposes a side effect of the orphan-PO defect: 44 lines appear to sit at the RFQ stage although SAP reports them as
closed. Their PO points to a requisition number that does not exist, so the link between requisition and PO is broken. The warehouse
cannot repair this safely on its own; it is reported for manual re-linking in SAP, a typical data-stewardship task. Every repair the
warehouse does make is still visible through its flag.

## 4. The star schema

In [6]:
marts = query("""
    select schema_name as layer, table_name, estimated_size as rows
    from duckdb_tables() where schema_name in ('marts', 'kpis') order by 1, 2
""")
grain = {
    "fct_requisition_line": "one requisition line", "fct_status_event": "one line x one event", "fct_payment": "one payment",
    "fct_work_order": "one work order", "fct_backlog_weekly": "week x section x supplier origin",
    "fct_weekly_section": "week x section", "ml_features_weekly": "section x origin week", "dim_supplier": "one supplier",
    "dim_material": "one material", "dim_section": "one section", "dim_equipment": "one equipment unit", "dim_date": "one day",
    "dim_purchasing_group": "one buying team", "kpi_monthly_company": "month x KPI",
}
marts["grain"] = marts.table_name.map(grain)
display(marts)

,layer,table_name,rows,grain
0,kpis,kpi_monthly_company,1055,month x KPI
1,marts,dim_date,1857,one day
2,marts,dim_equipment,206,one equipment unit
3,marts,dim_material,1500,one material
4,marts,dim_purchasing_group,6,one buying team
5,marts,dim_section,12,one section
6,marts,dim_supplier,60,one supplier
7,marts,fct_backlog_weekly,9984,week x section x supplier origin
8,marts,fct_payment,44443,one payment
9,marts,fct_requisition_line,34973,one requisition line


`fct_requisition_line` sits at the centre. Dimensions describe suppliers, materials, sections, equipment and dates. Three
aggregated facts serve specific consumers: the weekly backlog (stock and flow), the weekly section table (forecasting) and the
feature store (machine learning).

## 5. Testing strategy

Tests are what make the numbers trustworthy:

In [7]:
import json
manifest = json.loads((ROOT / "warehouse" / "target" / "manifest.json").read_text())
tests = [n for n in manifest["nodes"].values() if n["resource_type"] == "test"]
kinds = pd.Series([t.get("test_metadata", {}).get("name", "singular (custom SQL)") for t in tests]).value_counts()
display(kinds.rename("tests").to_frame())
print((ROOT / "warehouse" / "tests" / "assert_backlog_balances.sql").read_text())

,tests
not_null,15
unique,13
accepted_values,5
singular (custom SQL),4
dbt_utils_unique_combination,3
relationships,2


-- Stock-and-flow check: open(t) = open(t-1) + created(t) - received(t) - cancelled(t)
with w as (
    select week_start,
           sum(open_lines) as open_lines, sum(created_lines) as created,
           sum(received_lines) as received, sum(cancelled_lines) as cancelled
    from {{ ref('fct_backlog_weekly') }}
    group by week_start
),
chk as (
    select *, lag(open_lines) over (order by week_start) as prev_open from w
)
select * from chk
where prev_open is not null and open_lines <> prev_open + created - received - cancelled



The most valuable test is the **stock-and-flow identity**: every week, open backlog must equal last week's backlog plus new lines minus
receipts minus cancellations. If any join duplicated or lost a line, this test would fail immediately.

## 6. The governed KPI layer

KPI definitions, targets, thresholds and owners live in one seed (`kpi_catalog.csv`). The KPI model reads it, so a target changes in one
place and the dashboard follows.

In [8]:
latest = query("""
    select kpi_id, kpi_name, area, unit, round(kpi_value, 3) as value, target, status
    from kpis.kpi_monthly_company
    where month_start = (select max(month_start) from kpis.kpi_monthly_company)
    order by kpi_id
""")
display(latest)
print(latest.status.value_counts().to_string())

,kpi_id,kpi_name,area,unit,value,target,status
0,K01,PR-to-Receipt Cycle Time P50,North Star,days,62.000,60.000,At risk
1,K02,Critical Parts On-Time Rate,North Star,%,0.405,0.550,Off target
2,K03,Open Backlog Lines,Backlog,lines,"2,009.000","1,800.000",At risk
3,K04,Open Backlog Value,Backlog,USD,"5,630,693.450","4,000,000.000",Off target
4,K05,Backlog Aged over 90 Days,Backlog,%,0.268,0.300,On target
5,K06,Backlog Aged over 150 Days,Backlog,%,0.081,0.100,On target
6,K07,Backlog Burn Rate,Backlog,ratio,1.331,1.000,On target
7,K08,PR Release to RFQ Time P50,Cycle Time,days,5.000,5.000,On target
8,K09,PO Approval Time P50 (MD level),Cycle Time,days,4.500,3.000,At risk
9,K10,PO to Delivery Lead Time P50,Cycle Time,days,40.000,40.000,On target


status
On target     9
At risk       6
Off target    6
Monitor       1


**What we see (December 2025).** Of 22 KPIs, 9 are on target, 6 at risk, 6 off target and cash out is monitored without a target:

* **On target:** release-to-RFQ time, PO-to-delivery lead time, aged backlog shares, burn rate, PO rejections, pending downpayments,
  price variance and data quality.
* **Off target:** OTIF (28% against 60%), lead-time variability, open backlog value, critical parts on time (41% against 55%) and work
  orders waiting for parts.

The pattern points at one theme the rest of the project explores: **imported parts drive delays, uncertainty and equipment waiting
time**, while the internal process (after the 2024 policy change) is mostly under control.

## 7. Documentation

`dbt docs generate` builds a searchable website with every model, column description, test and the lineage graph. It is published
with GitHub Pages (see the README).

## 8. Summary

* The audit rules are now **code**, versioned and tested (42 data tests on 33 models).
* One star schema serves every consumer: notebooks, dashboard, agent and machine-learning features.
* KPIs are governed in one place, with targets and owners.

**Next:** notebook 04 uses this warehouse to answer procurement questions with advanced SQL.